In [1]:
import torch
from transformers import BertTokenizer
from transformers import BertForQuestionAnswering

#### Question Answering

In [2]:
question = "Who built the Taj Mahal?"
context = "The Taj Mahal is a white marble mausoleum located in Agra, India. It was built by the Mughal emperor Shah Jahan in memory of his wife Mumtaz Mahal."

In [3]:
model_name = "bert-large-uncased-whole-word-masking-finetuned-squad"

In [4]:
tokenizer = BertTokenizer.from_pretrained(model_name)

In [5]:
model = BertForQuestionAnswering.from_pretrained(model_name)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking-finetuned-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


##### tokenize

In [6]:
input_data = tokenizer(question, context, return_tensors = "pt")
print(input_data)

{'input_ids': tensor([[  101,  2040,  2328,  1996, 11937,  3501, 27913,  1029,   102,  1996,
         11937,  3501, 27913,  2003,  1037,  2317,  7720, 19049,  2284,  1999,
         29542,  1010,  2634,  1012,  2009,  2001,  2328,  2011,  1996, 17877,
          3750,  7890, 14855,  4819,  1999,  3638,  1997,  2010,  2564, 12954,
          2696,  2480, 27913,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [7]:
token_ids = input_data['input_ids'][0]

tokens = tokenizer.convert_ids_to_tokens(token_ids)
print(tokens)

['[CLS]', 'who', 'built', 'the', 'ta', '##j', 'mahal', '?', '[SEP]', 'the', 'ta', '##j', 'mahal', 'is', 'a', 'white', 'marble', 'mausoleum', 'located', 'in', 'agra', ',', 'india', '.', 'it', 'was', 'built', 'by', 'the', 'mughal', 'emperor', 'shah', 'ja', '##han', 'in', 'memory', 'of', 'his', 'wife', 'mum', '##ta', '##z', 'mahal', '.', '[SEP]']


##### get answer

In [8]:
with torch.no_grad():
    output = model(**input_data) # '**' to unpack the dictionary

print(output)

QuestionAnsweringModelOutput(loss=None, start_logits=tensor([[-6.3423, -6.9085, -7.7661, -7.6475, -7.9541, -8.4500, -8.7055, -9.2556,
         -6.3422, -3.8414, -3.4623, -6.4962, -5.2042, -6.9183, -6.1597, -4.8712,
         -5.1130, -5.0149, -6.8240, -7.1107, -3.1558, -7.6673, -3.3268, -6.3420,
          0.2694, -3.5687, -0.5430, -0.7449,  4.9567,  5.8805,  3.6347,  5.9679,
         -2.0913, -2.5624, -4.5249, -4.6321, -7.5994, -3.6014, -5.3873, -2.3003,
         -7.3226, -7.6329, -5.3017, -6.3320, -6.3425]]), end_logits=tensor([[-0.9685, -6.6871, -6.9572, -7.4921, -7.4172, -6.3516, -5.7262, -6.4705,
         -0.9684, -5.8405, -5.4951, -4.0909, -3.1071, -6.7102, -7.1638, -6.5170,
         -4.7366, -3.5364, -6.1191, -6.9554, -3.0867, -5.5543, -2.7281, -0.9681,
         -5.3516, -5.0741, -4.1360, -3.9699, -2.9976, -0.0204,  1.4276, -0.4461,
         -0.4595,  7.9457, -1.1176, -2.2850, -4.6851, -3.7507, -2.3174, -4.8285,
         -5.7369, -4.8492,  0.4500,  2.1885, -0.9697]]), hidden_state

In [9]:
softmax = torch.nn.Softmax(dim = 1) # softmax layer

start_probs = softmax(output.start_logits)
end_probs = softmax(output.end_logits)

In [10]:
start_index = start_probs.argmax().item() # argmax(): index with max prob, item(): convert tensor to int
end_index = end_probs.argmax().item()

print(start_index, end_index)

31 33


In [11]:
answer = tokenizer.decode(token_ids[start_index : end_index + 1])

print(question)
print(answer)

Who built the Taj Mahal?
shah jahan
